In [ ]:
import os
import json
from dotenv import load_dotenv

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client import models

In [ ]:
load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qdrant_client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "renovation_docs"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

In [ ]:
OUT_OF_DOMAIN_MESSAGE = (
    "Я помогаю только с вопросами о ремонте и услугах компании."
)

CONTACTS_RESPONSE = (
    "Связаться с менеджером можно по телефону +7(499)372-47-47. "
    "Также можно оставить заявку в приложении или на сайте."
)

SERVICE_AREA_RESPONSE = (
    "В материалах есть информация о работе в Москве. По Московской области и другим регионам лучше уточнить у менеджера."
)

SERVICE_SCOPE_RESPONSE = (
    "В материалах основная информация касается ремонта квартир. По офисам, домам, коттеджам, складам и другим типам объектов лучше уточнить у менеджера."
)

INSTALLMENTS_RESPONSE = (
    "Информация о рассрочке в материалах явно не подтверждена. Лучше уточнить этот вопрос у менеджера."
)

ESTIMATE_RESPONSE = (
    "Примерный порядок цен можно понять предварительно, но без замера или детального описания точную сумму назвать нельзя. Она зависит от площади, состояния квартиры и состава работ."
)

MIN_RETRIEVAL_SCORE = 0.35

## Embedding query


In [ ]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=query,
    )
    return response.data[0].embedding

## Qdrant Retreival

In [ ]:
def retrieve_candidates(query: str, top_k: int = 8):
    query_vector = embed_query(query)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return response.points

In [ ]:
def retrieve_candidates_once(query: str, top_k: int = 8, source_mode: str = "all"):
    query_vector = embed_query(query)

    query_filter = None

    if source_mode == "pricing_only":
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="document_type",
                    match=models.MatchValue(value="pricing"),
                )
            ]
        )
    elif source_mode == "website_only":
        query_filter = models.Filter(
            must_not=[
                models.FieldCondition(
                    key="document_type",
                    match=models.MatchValue(value="pricing"),
                )
            ]
        )

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=query_filter,
    )

    return response.points

In [ ]:
def retrieve_candidates(query: str, top_k: int = 8):
    source_mode = "website_only" if is_broad_repair_question(query) else "all"

    candidates = retrieve_candidates_once(
        query=query,
        top_k=top_k,
        source_mode=source_mode,
    )

    if should_retry_with_rewrite(query, candidates):
        rewritten_query = rewrite_query_with_llm(query)

        candidates = retrieve_candidates_once(
            query=rewritten_query,
            top_k=top_k,
            source_mode=source_mode,
        )

    return candidates

## Context

In [ ]:
def build_context(points) -> str:
    parts = []

    for point in points:
        payload = point.payload or {}
        text = (payload.get("text") or "").strip()
        if text:
            parts.append(text)

    return "\n\n---\n\n".join(parts)

In [ ]:
def rerank_chunks(question: str, candidates, top_n: int = 3):
    scored = []

    for point in candidates:
        payload = point.payload or {}
        chunk_text = (payload.get("text") or "").strip()

        if not chunk_text:
            continue

        rerank_prompt = f"""
Вопрос:
{question}

Фрагмент:
{chunk_text}

Оцени, насколько этот фрагмент помогает ответить на вопрос пользователя.

Верни только одно число от 0 до 10:
0 — фрагмент не помогает
10 — фрагмент максимально полезен
""".strip()

        response = openai_client.responses.create(
            model=CHAT_MODEL,
            input=[
                {
                    "role": "system",
                    "content": "Ты оцениваешь релевантность фрагмента для ответа на вопрос. Верни только число от 0 до 10."
                },
                {
                    "role": "user",
                    "content": rerank_prompt,
                },
            ],
            max_output_tokens=16,
            temperature=0,
        )

        raw_score = (response.output_text or "").strip()

        try:
            score = float(raw_score.replace(",", "."))
        except ValueError:
            score = 0.0

        scored.append((score, point))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [point for score, point in scored[:top_n]]

In [ ]:
def is_broad_repair_question(question: str) -> bool:
    q = question.lower()

    broad_hints = [
        "квартира", "однушка", "двушка", "трешка", "студия",
        "ремонт квартиры", "стоимость ремонта", "сколько выйдет",
        "сколько будет стоить", "примерная стоимость",
        "сколько примерно", "капитальный ремонт", "дизайнерский ремонт",
        "вторич", "новострой",
        "ремонт кухни", "кухн",
        "ремонт ванной", "ванн",
        "ремонт санузла", "сануз",
    ]

    return any(hint in q for hint in broad_hints)


In [ ]:
def looks_like_repair_question(question: str) -> bool:
    q = question.lower()

    repair_hints = [
        "ремонт", "квартира", "однушка", "двушка", "трешка", "студия",
        "кухн", "ванн", "сануз", "новострой", "вторич",
        "смет", "стоим", "цена", "цен", "порядок цен",
        "материал", "дизайн", "демонтаж", "монтаж", "отделк",
        "замер", "замерщик", "договор", "гарант", "рассроч",
        "менеджер", "контакт", "связат", "написать", "заявк",
        "телефон", "номер", "ватсап", "whatsapp", "телега", "telegram",
        "под ключ", "чернов", "чистов", "электрик", "сантех",
        "офис", "коммерц", "нежил", "коттедж", "дом",
        "москва", "область", "регион", "без выезда",
        "работаете", "работаете ли", "где работаете", "по москве", "по области",
    ]

    return any(hint in q for hint in repair_hints)

In [ ]:
def is_contact_question(question: str) -> bool:
    q = question.lower()

    contact_hints = [
        "контакт", "контакты", "связаться", "связь",
        "менеджер", "телефон", "номер", "позвонить",
        "написать", "заявка", "заявку", "оставить заявку",
        "ватсап", "whatsapp", "телега", "telegram",
    ]

    return any(hint in q for hint in contact_hints)

In [ ]:
def answer_contact_question(question: str):
    return CONTACTS_RESPONSE, [], ""


INTENTS = {
    "contacts",
    "service_area",
    "service_scope",
    "payment_terms",
    "remote_estimate",
    "broad_pricing",
    "specific_pricing",
    "service_confirmation",
    "general_rag",
    "out_of_scope",
}


def classify_question(question: str) -> dict:
    prompt = f"""
Определи тип вопроса пользователя для ассистента компании по ремонту квартир.

Верни только JSON без пояснений:
{{"intent": "...", "rewritten_query": "..."}}

Допустимые intent:
- contacts: пользователь просит телефон, контакты, WhatsApp, Telegram, связь с менеджером, куда написать или оставить заявку.
- service_area: вопрос про Москву, область, регион, выезд, где компания работает.
- service_scope: вопрос про тип объекта: только квартиры или также офисы, дома, коттеджи, склады, коммерческие или нежилые помещения.
- payment_terms: вопрос про рассрочку, оплату, этапы оплаты, оплату по факту.
- remote_estimate: вопрос можно ли оценить цену без выезда, без замера, по фото, предварительно или узнать порядок цен.
- broad_pricing: общая стоимость ремонта помещения или квартиры: ремонт кухни, ванной, санузла, однушки, двушки, квартиры 40 м², порядок стоимости ремонта.
- specific_pricing: цена конкретной работы из прайса: демонтаж плитки, установка мойки, монтаж подрозетника, поклейка обоев и т.п.
- service_confirmation: вопрос выполняет ли компания конкретную услугу ремонта: дизайнерский ремонт, капитальный ремонт, ремонт в новостройке, ремонт ванной под ключ и т.п.
- general_rag: другой вопрос по ремонту и услугам компании.
- out_of_scope: вопрос не связан с ремонтом, отделкой, дизайном, ценами, сроками, контактами или услугами компании.

rewritten_query:
- для поиска перепиши вопрос формально и исправь опечатки;
- сохрани смысл;
- не добавляй факты;
- для contacts и out_of_scope можно оставить пустую строку.

Примеры:
"скок выйдет ремнот аднушки" -> {{"intent":"broad_pricing","rewritten_query":"стоимость ремонта однокомнатной квартиры"}}
"а вы ванну под ключ делаете или нет" -> {{"intent":"service_confirmation","rewritten_query":"ремонт ванной комнаты под ключ"}}
"можно без замера хотябы примерно понять цену" -> {{"intent":"remote_estimate","rewritten_query":"можно ли предварительно оценить стоимость ремонта без замера"}}
"сколько стоит демонтаж плитки" -> {{"intent":"specific_pricing","rewritten_query":"стоимость демонтажа плитки"}}
"объясни квантовую механику" -> {{"intent":"out_of_scope","rewritten_query":""}}

Вопрос пользователя:
{question}
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": "Ты классифицируешь вопросы для RAG. Верни только валидный JSON."},
            {"role": "user", "content": prompt},
        ],
        max_output_tokens=120,
        temperature=0,
    )

    raw = (response.output_text or "").strip()

    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        data = {"intent": "general_rag", "rewritten_query": question}

    intent = str(data.get("intent", "general_rag")).strip()
    if intent not in INTENTS:
        intent = "general_rag"

    rewritten_query = str(data.get("rewritten_query", "") or "").strip()

    return {
        "intent": intent,
        "rewritten_query": rewritten_query,
        "raw": raw,
    }


def source_mode_for_intent(intent: str) -> str:
    if intent == "specific_pricing":
        return "pricing_only"

    if intent in {
        "service_area",
        "service_scope",
        "payment_terms",
        "remote_estimate",
        "broad_pricing",
        "service_confirmation",
        "general_rag",
    }:
        return "website_only"

    return "all"


def instructions_for_intent(intent: str) -> str:
    common = "Не упоминай контекст, базу знаний или внутреннюю классификацию."

    if intent == "specific_pricing":
        return common + " Используй только точные цены и единицы измерения из найденных данных."

    if intent == "broad_pricing":
        return common + " Если есть цена за м² и пользователь указал площадь, посчитай ориентир. Если площадь не указана, не придумывай метраж. Не используй цены отдельных работ как цену всего ремонта."

    if intent == "remote_estimate":
        return common + " Объясни, что предварительный ориентир возможен, но точность зависит от площади, состояния помещения и состава работ. Не используй цены отдельных работ как общий расчет."

    if intent == "service_confirmation":
        return common + " Ответь, выполняется ли такой вид работ. Не называй цены. Если явного подтверждения нет, предложи уточнить у менеджера."

    if intent == "service_scope":
        return common + " Не отвечай уверенно да или нет по объектам вне квартир, если это явно не подтверждено. Лучше предложи уточнить у менеджера."

    if intent == "service_area":
        return common + " По географии работы отвечай только по явно подтвержденным данным. Если регион не подтвержден, предложи уточнить у менеджера."

    if intent == "payment_terms":
        return common + " По оплате и рассрочке отвечай только по явно подтвержденным данным. Если условий нет, предложи уточнить у менеджера."

    return common


In [ ]:
def is_pricing_point(point) -> bool:
    payload = point.payload or {}
    return payload.get("document_type") == "pricing"


def should_retry_with_rewrite(question: str, candidates) -> bool:

    if not looks_like_repair_question(question):
        return False

    if not candidates:
        return True

    top_score = candidates[0].score or 0.0

    if top_score < 0.35:
        return True

    if is_broad_repair_question(question):
        top3 = candidates[:3]
        pricing_count = sum(1 for point in top3 if is_pricing_point(point))

        if pricing_count >= 2:
            return True

        if top_score < 0.52:
            return True

    return False

## Prompt

In [ ]:
SYSTEM_PROMPT = """
Ты — помощник компании по ремонту квартир.

Правила:
1. Отвечай только по переданному контексту.
2. Если в контексте есть точная цена, срок, гарантия, формат работы или контакт — используй их в ответе.
3. Если вопрос про стоимость, считай только из явных данных в контексте. Не придумывай свои цифры, площади, проценты, сроки или диапазоны.
4. Если в контексте есть цена за м² и пользователь указал площадь, можешь посчитать примерную стоимость.
5. Если в контексте есть цена за м², но площадь не указана, можно сказать только, что стоимость считается от цены за м², без выдуманных площадей.
6. Если услуга, тип объекта, регион, рассрочка, замер, закупка материалов или другой факт не подтвержден в контексте явно, не отвечай уверенно "да" или "нет". Вместо этого скажи, что это лучше уточнить у менеджера.
7. Если информации недостаточно, скажи по-человечески, что лучше уточнить детали у менеджера.
8. Если вопрос вообще не связан с ремонтом и услугами компании, коротко скажи, что ты помогаешь только по вопросам ремонта и услуг компании.
9. Не используй общие знания, если их нет в контексте.
10. Не упоминай базу знаний.
11. Отвечай по-русски, коротко и естественно, максимум 4-5 предложений.
12. Если вопрос про общую стоимость ремонта, порядок цен или оценку без выезда, не используй отдельные цены на конкретные работы как ответ на общий вопрос.
13. Не называй компанию по имени, если пользователь сам об этом не спрашивал.
14. Не перефразируй факты из контекста так, чтобы менялся смысл. Например, "фиксированная смета" не означает "фиксированная цена".
15. Не комментируй сам вопрос пользователя и не говори фразы вроде "в вашем вопросе есть упоминание".
16. Не ссылайся на контекст в явном виде и не говори фразы вроде "в контексте указано" или "в контексте есть".
""".strip()


## Rewrite query

In [ ]:
def rewrite_query_with_llm(question: str) -> str:
    prompt = f"""
Перепиши вопрос пользователя в нормальный поисковый запрос для базы знаний по ремонту квартир.

Правила:
1. Сохрани исходный смысл.
2. Исправь разговорные слова и опечатки.
3. Замени бытовые формулировки на более стандартные.
4. Не добавляй новых фактов.
5. Верни только одну короткую переписанную фразу, без пояснений.

Вопрос:
{question}
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {
                "role": "system",
                "content": "Ты переписываешь пользовательский запрос для поиска по базе знаний. Верни только переписанный запрос.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_output_tokens=40,
        temperature=0,
    )

    rewritten = (response.output_text or "").strip()
    return rewritten or question

## Generator

In [ ]:
def answer_with_rag(question: str, top_k: int = 8, top_n: int = 3):
    route = classify_question(question)
    intent = route["intent"]

    if intent == "out_of_scope":
        return OUT_OF_DOMAIN_MESSAGE, [], ""

    if intent == "contacts":
        return answer_contact_question(question)

    search_query = route["rewritten_query"] or question
    source_mode = source_mode_for_intent(intent)

    candidates = retrieve_candidates_once(
        query=search_query,
        top_k=top_k,
        source_mode=source_mode,
    )

    if not candidates:
        return "По этому вопросу лучше уточнить детали у менеджера.", [], ""

    best_score = candidates[0].score or 0.0
    if best_score < MIN_RETRIEVAL_SCORE:
        return "По этому вопросу лучше уточнить детали у менеджера.", candidates, ""

    best_points = rerank_chunks(question, candidates, top_n=top_n)

    if not best_points:
        return "По этому вопросу лучше уточнить детали у менеджера.", [], ""

    context = build_context(best_points)
    intent_instructions = instructions_for_intent(intent)

    user_prompt = f"""
Вопрос пользователя:
{question}

Нормализованный поисковый запрос:
{search_query}

Тип вопроса:
{intent}

Контекст:
{context}

Правила для этого типа вопроса:
{intent_instructions}

Ответь коротко, естественно и только по найденным данным.
Если данных недостаточно, не выдумывай и предложи уточнить у менеджера.
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, best_points, context


In [ ]:
def answer_with_baseline(question: str, top_k: int = 5):
    points = retrieve_candidates(question, top_k=top_k)
    context = build_context(points[:top_k])

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}

Ответь по правилам.
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, points[:top_k], context

In [ ]:
def debug_intent(question: str):
    route = classify_question(question)
    print("QUESTION:", question)
    print("INTENT:", route["intent"])
    print("REWRITTEN:", route["rewritten_query"])
    print("RAW:", route["raw"])
    return route


In [ ]:
def debug_rewrite_flow(question: str, top_k: int = 8):
    source_mode = "website_only" if is_broad_repair_question(question) else "all"
    first = retrieve_candidates_once(question, top_k=top_k, source_mode=source_mode)

    print("QUESTION:", question)
    print("SOURCE MODE:", source_mode)
    print("FIRST TOP SCORE:", first[0].score if first else None)
    print("REWRITE NEEDED:", should_retry_with_rewrite(question, first))

    if should_retry_with_rewrite(question, first):
        rewritten = rewrite_query_with_llm(question)
        print("REWRITTEN:", rewritten)

        second = retrieve_candidates_once(rewritten, top_k=top_k, source_mode=source_mode)
        print("SECOND TOP SCORE:", second[0].score if second else None)
        return first, second

    return first, first

# Tests

In [ ]:
def print_points(points, max_chars: int = 500):
    for i, point in enumerate(points, start=1):
        payload = point.payload or {}
        print("=" * 100)
        print(f"RESULT #{i}")
        print(f"score: {getattr(point, 'score', None)}")
        print(f"title: {payload.get('title')}")
        print(f"url: {payload.get('url')}")
        print()
        print((payload.get("text") or "")[:max_chars])
        print()

In [ ]:
def compare_answers(question: str, baseline_top_k: int = 5, rerank_top_k: int = 8, rerank_top_n: int = 3):
    print("#" * 120)
    print("QUESTION:", question)
    print()

    baseline_answer, baseline_points, _ = answer_with_baseline(question, top_k=baseline_top_k)
    print("BASELINE ANSWER:")
    print(baseline_answer)
    print()

    rerank_answer, rerank_points, _ = answer_with_rag(question, top_k=rerank_top_k, top_n=rerank_top_n)
    print("RERANK ANSWER:")
    print(rerank_answer)
    print()

    print("BASELINE POINTS:")
    print_points(baseline_points, max_chars=350)

    print("RERANK POINTS:")
    print_points(rerank_points, max_chars=350)

In [ ]:
test_questions = [
    "Сколько примерно стоит ремонт квартиры 40 квадратов?",
    "Хочу понять, во сколько выйдет ремонт однушки",
    "Сколько стоит сделать ремонт на кухне?",
    "А ванную под ключ делаете?",
    "Есть ли у вас дизайнерский ремонт?",
    "Вы делаете ремонт в новостройке?",
    "Сколько по времени обычно идет ремонт квартиры?",
    "Можно примерно понять цену без выезда?",
    "От чего вообще зависит стоимость ремонта?",
    "Есть ли у вас гарантия на работы?",
    "Вы по договору работаете?",
    "Можно у вас заказать ремонт + дизайн сразу?",
    "Делаете ли вы капитальный ремонт?",
    "Скажите пожалуйста, вы только квартиры делаете?",
    "А если вторичка, вы с такими квартирами работаете?",
    "Сколько будет стоить ремонт двушки 54 метра?",
    "Можно примерную цену на ремонт кухни 12 метров?",
    "А если квартира убитая совсем, цена сильно выше будет?",
    "Сколько выйдет черновой ремонт?",
    "Можно ли по фото примерно посчитать стоимость?",
    "Есть ли у вас рассрочка?",
    "Работаете ли вы поэтапно или только под ключ?",
    "Можно ли сделать сначала санузел, а потом остальное?",
    "Вы закупкой материалов занимаетесь?",
    "Кто у вас контролирует ремонт на объекте?",
    "Дай контакты менеджера",
    "Как с вами связаться?",
    "Можно номер телефона",
    "Куда написать чтобы обсудить ремонт?",
    "А ватсап есть?",
    "С кем можно поговорить по расчету стоимости?",
    "Хочу связаться с менеджером",
    "Где оставить заявку?",
    "Как вам написать?",
    "Есть у вас телега или вотсап?",
    "Вы делаете ремонт складских помещений?",
    "А офисы ремонтируете?",
    "С коммерцией работаете?",
    "Можно у вас заказать ремонт дома, не квартиры?",
    "Вы делаете только квартиры или коттеджи тоже?",
    "А нежилые помещения берете в работу?",
    "Можно у вас заказать просто дизайн без ремонта?",
    "Только Москва или область тоже?",
    "Вы сами материалы покупаете или это на клиенте?",
    "Можно вызвать замерщика?",
    "Какая погода в Москве?",
    "Объясни квантовую механику простыми словами",
    "Напиши код на Python для телеграм бота",
    "Кто выиграл чемпионат мира?",
    "Как приготовить карбонару?",
    "Посоветуй фильм на вечер",
    "Сколько будет 245 умножить на 17?",
    "Расскажи анекдот",
    "Что лучше, айфон или самсунг?",
    "Помоги выбрать ноутбук",
    "скок стоит ремонт кухни примерно",
    "а вы ванну под ключ делаете или нет?",
    "мне нужен ремнот квартиры 40м2 сколько это выйдет",
    "если у меня вторичка убитая совсем это сильно дороже?",
    "можно без замера хотябы примерно понять цену",
    "дайте номер менеджера пж",
    "а с вами можно в ватсапе списаться?",
    "а офисы вы тоже делаете или тока квартиры?",
    "а если я просто хочу узнать порядок цен",
    "вы вобще по москве только работаете?",
]

In [ ]:
check_questions = [
    "Можно примерно понять цену без выезда?",
    "Есть ли у вас рассрочка?",
    "Как с вами связаться?",
    "А ватсап есть?",
    "Как вам написать?",
    "Только Москва или область тоже?",
    "а если я просто хочу узнать порядок цен",
    "вы вобще по москве только работаете?",
    "А офисы ремонтируете?",
    "Можно у вас заказать ремонт дома, не квартиры?",
    "Дай контакты менеджера",
    "Как с вами связаться?",
    "Можно номер телефона",
    "А ватсап есть?",
    "Как вам написать?",
    "Есть у вас телега или вотсап?",
]

In [ ]:
for q in test_questions:
    answer, _, _ = answer_with_rag(q)
    print(f"Question: {q} \nAnswer: {answer}\n")

## Debug questions

In [ ]:
debug_questions = [
    "Хочу понять, во сколько выйдет ремонт однушки",
    "скок стоит ремонт двушки",
    "если вторичка убитая совсем это сильно дороже?",
    "Сколько стоит демонтаж плитки?",
    "Объясни квантовую механику",
]

In [ ]:
for q in debug_questions:
    print("=" * 100)
    debug_rewrite_flow(q)
    print()

## Production RAG check

Этот блок проверяет уже рабочий код из `app/rag.py`, а не старые функции из ноутбука.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/himynameisartem/Documents/Development/ML/Renovation_calculator_AI_assistant")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.rag import RenovationRAG

rag = RenovationRAG()


In [ ]:
result = rag.answer("Сколько примерно стоит ремонт квартиры 40 квадратов?")

print("ANSWER:", result.answer)
print("INTENT:", result.intent)
print("REWRITTEN:", result.rewritten_query)
print("POINTS:", len(result.points))


In [3]:
test_questions = [
    "Сколько примерно стоит ремонт квартиры 40 квадратов?",
    # "Хочу понять, во сколько выйдет ремонт однушки",
    "скок стоит ремонт двушки",
    # "если вторичка убитая совсем это сильно дороже?",
    "Сколько стоит демонтаж плитки?",
    # "Сколько стоит сделать ремонт на кухне?",
    "А ванную под ключ делаете?",
    "Есть ли у вас дизайнерский ремонт?",
    # "Можно примерно понять цену без выезда?",
    "можно без замера хотябы примерно понять цену",
    # "Можно ли по фото примерно посчитать стоимость?",
    "Есть ли у вас рассрочка?",
    "Как с вами связаться?",
    # "А ватсап есть?",
    # "Как вам написать?",
    # "Дай контакты менеджера",
    "Только Москва или область тоже?",
    "вы вобще по москве только работаете?",
    "Скажите пожалуйста, вы только квартиры делаете?",
    "А офисы ремонтируете?",
    # "Можно у вас заказать ремонт дома, не квартиры?",
    # "Вы делаете ремонт складских помещений?",
    "Можно вызвать замерщика?",
    "Зависит ли расположение квартиры на стоимость работ?",
    "Какие работы входят в капитальный ремонт?",
    # "Сколько длится ремонт?",
    "Работаете с новостройками?",
    "Можно заказать дизайн-проект?",
    "Что входит в черновой ремонт?",
    "Объясни квантовую механику",
]


In [4]:
results = []

for index, question in enumerate(test_questions, start=1):
    result = rag.answer_fast(question)

    results.append({
        "question": question,
        "answer": result.answer,
        "intent": result.intent,
        "rewritten_query": result.rewritten_query,
        "points_count": len(result.points),
    })

    print("=" * 100)
    print(f"#{index}")
    print("QUESTION:", question)
    print("INTENT:", result.intent)
    print("REWRITTEN:", result.rewritten_query)
    print("POINTS:", len(result.points))
    print("ANSWER:", result.answer)


#1
QUESTION: Сколько примерно стоит ремонт квартиры 40 квадратов?
INTENT: general_rag
REWRITTEN: Сколько примерно стоит ремонт квартиры 40 квадратов?
POINTS: 3
ANSWER: Примерная стоимость ремонта квартиры площадью 40 м² — от 600 000 ₽ (40 × 15 000 ₽). Точная цена зависит от состояния квартиры, состава работ, материалов и пожеланий. Для точной сметы нужен замер.
#2
QUESTION: скок стоит ремонт двушки
INTENT: general_rag
REWRITTEN: скок стоит ремонт двушки
POINTS: 3
ANSWER: Стоимость ремонта двушки начинается от 15 000 ₽ за м². Точная цена зависит от площади, состояния квартиры, состава работ, материалов и пожеланий. Для расчёта точной сметы нужен замер.
#3
QUESTION: Сколько стоит демонтаж плитки?
INTENT: general_rag
REWRITTEN: Сколько стоит демонтаж плитки?
POINTS: 3
ANSWER: Стоимость демонтажа плитки зависит от места выполнения работ:
* демонтаж керамической плитки со стен — 230 ₽ за м²;
* демонтаж плитки с полов — 180 ₽ за м²;
* демонтаж порога из плитки — 550 ₽ за штуку.
#4
QUESTION: 

In [8]:
result = rag.answer_fast("Объясни квантовую механику")
print(result.answer)

Я помогаю только с вопросами о ремонте и услугах компании.
